# BookAssembler на Google Colab

Полный пайплайн перевода технических книг с GPU-ускорением.

Все файлы хранятся на **Google Drive** — PDF книги, переводы, кэш и результаты.
Если Colab перезапустится, прогресс не потеряется — `--resume` продолжит с места остановки.

**Требования:** GPU runtime (Runtime → Change runtime type → T4 GPU)

## 1. Подключение Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive подключён")

## 2. Установка Ollama + модель

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
#@title Модель { display-mode: "form" }
MODEL = "llama3.1:8b" #@param ["llama3.1:8b", "gemma3:12b", "qwen2.5:14b", "phi4:14b", "mistral:7b"]

In [ ]:
import subprocess, time, os

env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0"
proc = subprocess.Popen(["ollama", "serve"], env=env,
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
print("Ollama запущен")

!ollama pull {MODEL}
print(f"\nМодель {MODEL} готова")
!nvidia-smi | head -4

## 3. Настройка рабочей директории на Google Drive

Структура на Drive:
```
Google Drive/
  BookAssembler/
    mybook.pdf              ← PDF книги (положите сюда)
    chapters.yaml           ← структура книги
    book_profile.yaml       ← автоопределяется
    glossary.json           ← словарь (опционально)
    cache/text/             ← извлечённый текст
    cache/state/            ← состояние пайплайна
    claude_translations/    ← переводы
    latex_output/           ← собранные .tex файлы
```

In [ ]:
#@title Путь к проекту на Google Drive { display-mode: "form" }
GDRIVE_PROJECT = "/content/drive/MyDrive/BookAssembler" #@param {type:"string"}

In [ ]:
os.makedirs(GDRIVE_PROJECT, exist_ok=True)

# Нужные поддиректории
for d in ["cache/text", "cache/state", "cache/diagram_analysis",
          "claude_translations", "latex_output", "figures"]:
    os.makedirs(os.path.join(GDRIVE_PROJECT, d), exist_ok=True)

print(f"Рабочая директория: {GDRIVE_PROJECT}")
print()
print("Содержимое:")
!ls -la {GDRIVE_PROJECT}/

## 4. Установка BookAssembler

In [ ]:
#@title Источник кода { display-mode: "form" }
REPO_URL = "" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}

In [ ]:
# Клонируем код в /content (локально, не на Drive — код не нужно сохранять)
if REPO_URL:
    !git clone -b {BRANCH} {REPO_URL} /content/bookassembler_src 2>/dev/null || \
        (cd /content/bookassembler_src && git pull)
    src_dir = "/content/bookassembler_src"
else:
    # Код тоже на Drive
    src_dir = GDRIVE_PROJECT

!pip install -q pymupdf opencv-python-headless pyyaml openai numpy 2>&1 | tail -3

# Проверка
assert os.path.exists(os.path.join(src_dir, "src", "pipeline.py")), \
    f"pipeline.py не найден в {src_dir}/src/ — проверьте путь к репозиторию"
print(f"BookAssembler: {src_dir}")

## 5. Конфигурация книги

Положите PDF в папку проекта на Drive и создайте `chapters.yaml`.

Если `chapters.yaml` уже есть — эта ячейка покажет его содержимое.

In [ ]:
chapters_path = os.path.join(GDRIVE_PROJECT, "chapters.yaml")

if os.path.exists(chapters_path):
    print("chapters.yaml уже существует:")
    print()
    with open(chapters_path) as f:
        print(f.read())
else:
    # Найти PDF-файлы в проекте
    import glob
    pdfs = glob.glob(os.path.join(GDRIVE_PROJECT, "*.pdf"))
    if pdfs:
        pdf_name = os.path.basename(pdfs[0])
        print(f"Найден PDF: {pdf_name}")
    else:
        pdf_name = "ВСТАВЬТЕ_ИМЯ_PDF.pdf"
        print("⚠️  PDF не найден! Положите PDF-файл в папку проекта на Google Drive.")

    template = f"""book:
  title: "Название книги"
  pdf: "{pdf_name}"
  target_lang: ru

chapters:
  1:
    pages: [10, 45]
    title: "Introduction"
  2:
    pages: [46, 120]
    title: "Chapter 2"
"""
    with open(chapters_path, "w") as f:
        f.write(template)
    print(f"\nСоздан шаблон chapters.yaml — ОТРЕДАКТИРУЙТЕ его!")
    print("Откройте файл на Google Drive или дважды кликните в панели слева.")
    print()
    print(template)

## 6. Запуск пайплайна

Все результаты пишутся сразу на Google Drive.
При перезапуске Colab — повторите шаги 1-4, затем запустите с `--resume`.

In [ ]:
#@title Параметры запуска { display-mode: "form" }
CHAPTER_FROM = 1 #@param {type:"integer"}
CHAPTER_TO = 1 #@param {type:"integer"}
STAGE = "" #@param ["", "extract", "detect", "manifest", "figures", "translate", "autofix", "validate", "build"]
RESUME = True #@param {type:"boolean"}

In [ ]:
# Симлинк src → рабочую директорию, чтобы pipeline.py видел данные
for link_name in ["cache", "claude_translations", "latex_output", "figures",
                  "chapters.yaml", "book_profile.yaml", "glossary.json"]:
    src_path = os.path.join(GDRIVE_PROJECT, link_name)
    dst_path = os.path.join(src_dir, link_name)
    if os.path.exists(src_path) and not os.path.exists(dst_path):
        os.symlink(src_path, dst_path)

# PDF-файл тоже линкуем
import glob
for pdf in glob.glob(os.path.join(GDRIVE_PROJECT, "*.pdf")):
    dst = os.path.join(src_dir, os.path.basename(pdf))
    if not os.path.exists(dst):
        os.symlink(pdf, dst)

# .env
env_path = os.path.join(src_dir, ".env")
with open(env_path, "w") as f:
    f.write(f"TRANSLATE_MODE=api\n")
    f.write(f"AI_PROVIDER=openai\n")
    f.write(f"AI_API_KEY=ollama\n")
    f.write(f"AI_BASE_URL=http://localhost:11434/v1\n")
    f.write(f"AI_MODEL={MODEL}\n")

# Запуск
stage_arg = f"--stage {STAGE}" if STAGE else ""
resume_arg = "--resume" if RESUME else ""

%cd {src_dir}

for ch in range(CHAPTER_FROM, CHAPTER_TO + 1):
    print(f"\n{'='*60}")
    print(f"  Глава {ch}")
    print(f"{'='*60}\n")
    !python3 src/pipeline.py --chapter {ch} {stage_arg} {resume_arg}

## 7. Статус

In [ ]:
%cd {src_dir}
!python3 src/pipeline.py --list

## 8. Просмотр результатов на Drive

Все файлы уже на Google Drive — ничего скачивать не нужно.

```
Google Drive/BookAssembler/
  claude_translations/     ← JSON с переводами
  latex_output/            ← готовые .tex файлы
  cache/state/             ← прогресс пайплайна
```

In [ ]:
print("=== Переводы ===")
!ls -lh {GDRIVE_PROJECT}/claude_translations/ 2>/dev/null || echo "Нет переводов"
print("\n=== LaTeX ===")
!ls -lh {GDRIVE_PROJECT}/latex_output/ 2>/dev/null || echo "Нет LaTeX"
print("\n=== Состояние ===")
!ls -lh {GDRIVE_PROJECT}/cache/state/ 2>/dev/null || echo "Нет состояния"

## Диагностика

In [ ]:
!nvidia-smi
!echo "---"
!ollama list
!echo "---"
!df -h /content/drive/MyDrive/ | tail -1